# GALAGA · 갤러그 — Colab 아케이드

**1 → 2 → 3번 코드 셀을 차례로 실행하면 브라우저에서 원작 갤러그를 실행할 수 있습니다.**

준비할 파일:
- `Galaga_resources.zip` — 압축을 풀지 않고 1번 셀에서 업로드합니다.
- **사용 권한이 있는 `galaga.zip` ROM** — 이 결과물에는 게임 ROM이 포함되지 않습니다. 1번 셀에서 별도로 업로드합니다.
- 분리된 ROM 세트에서 `51xx.bin`, `54xx.bin`이 빠졌다면 `namco51.zip`, `namco54.zip`도 함께 선택합니다.

대상은 **1981 Namco Galaga (Namco rev. B)**입니다. NES/패미컴용 ROM이나 Galaga '88은 사용할 수 없습니다. 파일 내용의 크기·해시가 다르면 원인을 안내합니다.

**실행 순서:** 1번 ZIP/ROM 업로드 → 2번 화면 준비 → 3번 주소 발급 → **게임 열기 → 게임 시작 → 기판 검사 대기 → 동전 넣기 → 1인 시작**.

CPU 런타임을 사용합니다. GPU, API 키, 유료 API, PC 프로그램 설치는 필요 없습니다. Colab 런타임 연결을 유지해야 외부 주소에 접속할 수 있습니다.

게임 연산은 접속한 브라우저에서 실행합니다. 기본은 제작 환경에서 변환한 **JavaScript 엔진**이며, 실행 정보에서 **WebAssembly**로 전환할 수 있습니다. 전환/새로고침하면 현재 게임 진행은 초기화됩니다.


## 1. 리소스 준비
▶ 버튼을 눌러 리소스 ZIP을 선택한 후 ROM ZIP을 선택합니다. 한 번에 모두 업로드해도 됩니다. 정상 완료 메시지는 **리소스 검증 완료**입니다. 이미 검증된 파일은 셀을 다시 실행해도 재사용합니다.

ROM 16개 구성 파일은 ZIP의 `rom-required.json`에 공개되어 있습니다. 이름이 달라도 내용이 정확히 일치하면 표준 파일명으로 정리합니다. 실패 시 누락/버전/해시 불일치를 구분한 메시지를 확인하세요.

In [ ]:
# 1번 — 리소스와 ROM 준비
from pathlib import Path, PurePosixPath
import hashlib, importlib.util, io, json, os, shutil, tempfile, zipfile
from google.colab import files
ROOT = Path('/content/galaga_github')
EXPECTED_MANIFEST_SHA256 = '4523a1983bf963c851fd62abaafd1e1f9ef7d00518eab0fed0b78e17dcb7c609'
def sha(data): return hashlib.sha256(data).hexdigest()
def verified_existing():
    try:
        raw=(ROOT/'source.json').read_bytes()
        if sha(raw)!=EXPECTED_MANIFEST_SHA256: return False
        info=json.loads(raw)
        return all((ROOT/n).is_file() and (ROOT/n).stat().st_size==r['size'] and sha((ROOT/n).read_bytes())==r['sha256'] for n,r in info['files'].items())
    except (OSError, ValueError, KeyError): return False
uploads={}
if verified_existing():
    print('기존 리소스 검증 완료 · 다시 업로드하지 않아도 됩니다.')
else:
    print('Galaga_resources.zip을 압축을 풀지 않고 선택하세요. ROM ZIP도 함께 선택할 수 있습니다.')
    uploads=files.upload()
    selected=[]
    for name,data in uploads.items():
        try:
            with zipfile.ZipFile(io.BytesIO(data)) as z:
                if 'source.json' in z.namelist(): selected.append(name)
        except zipfile.BadZipFile: pass
    if len(selected)!=1: raise ValueError('리소스 ZIP을 한 개 선택해야 합니다. Galaga_resources.zip인지 확인하고 1번 셀을 다시 실행하세요.')
    data=uploads.pop(selected[0])
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        entries=z.infolist()
        if len(entries)>200 or sum(i.file_size for i in entries)>100_000_000: raise ValueError('리소스 ZIP의 크기 또는 파일 수가 비정상적입니다.')
        names=[i.filename for i in entries]
        if len(names)!=len(set(names)):raise ValueError('ZIP에 중복 경로가 있습니다.')
        for i in entries:
            p=PurePosixPath(i.filename)
            if p.is_absolute() or '..' in p.parts or '\\' in i.filename or i.flag_bits&1 or (i.external_attr>>16)&0o170000==0o120000:
                raise ValueError('허용하지 않는 ZIP 내부 경로/암호화/링크: '+i.filename)
        raw=z.read('source.json')
        if sha(raw)!=EXPECTED_MANIFEST_SHA256:raise ValueError('source.json 검증값 불일치. 이 노트북과 함께 제공된 리소스 ZIP을 사용하세요.')
        manifest=json.loads(raw)
        expected=set(manifest['files'])|{'source.json'}
        if set(names)!=expected:raise ValueError('리소스 ZIP 파일 목록이 다릅니다. 누락 또는 추가된 파일이 있습니다.')
        with tempfile.TemporaryDirectory(prefix='galaga-stage-',dir='/content') as temp:
            staged=Path(temp)
            for name in names:
                content=z.read(name)
                if name!='source.json':
                    record=manifest['files'][name]
                    if len(content)!=record['size']:raise ValueError('파일 크기 불일치: '+name)
                    if sha(content)!=record['sha256']:raise ValueError('SHA-256 불일치: '+name)
                target=staged/name;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(content)
            ROOT.mkdir(parents=True,exist_ok=True)
            for name in names:
                dest=ROOT/name;dest.parent.mkdir(parents=True,exist_ok=True);os.replace(staged/name,dest)
    print('실행엔진·리소스 SHA-256 검증 완료')
spec=importlib.util.spec_from_file_location('galaga_support',ROOT/'colab_support.py')
galaga_support=importlib.util.module_from_spec(spec);spec.loader.exec_module(galaga_support)
if uploads:
    result=galaga_support.normalize_rom(uploads)
    print(f"ROM 구성 {len(result['files'])}개 검증 완료")
elif galaga_support.valid_existing_rom():
    print('기존 ROM 검증 완료 · 다시 업로드하지 않아도 됩니다.')
else:
    print('사용 권한이 있는 galaga.zip을 선택하세요. 필요 시 namco51.zip / namco54.zip도 함께 선택합니다.')
    uploaded_roms=files.upload()
    if not uploaded_roms:raise ValueError('ROM을 선택하지 않았습니다. 게임 실행에는 ROM이 필요합니다. 1번 셀을 다시 실행하세요.')
    result=galaga_support.normalize_rom(uploaded_roms)
    print(f"ROM 구성 {len(result['files'])}개 검증 완료")
print('리소스 검증 완료 — 2번 셀을 실행하세요.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 실행 코드 준비
▶ 버튼을 누르면 게임 화면과 입력·사운드 제어 코드를 작성합니다. 정상 완료 메시지는 **HTML/CSS/JavaScript 준비 완료**입니다. 대용량 엔진은 노트북 본문에 넣지 않고 리소스 ZIP에서 불러옵니다.

In [ ]:
# 2번 — HTML / CSS / JavaScript 준비
from pathlib import Path
ROOT=Path('/content/galaga_github')
if not (ROOT/'rom-validation.json').is_file():
    raise RuntimeError('1번 셀에서 ROM 검증을 먼저 완료하세요.')
(ROOT/'index.html').write_text('<!DOCTYPE html>\n<html lang="ko"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>GALAGA · 갤러그 아케이드</title><link rel="stylesheet" href="style.css"></head>\n<body><main><header><div><span class="eyebrow">1981 / NAMCO / ARCADE</span><h1>GALAGA<span>갤러그</span></h1></div><span class="badge">INSERT COIN · PLAY AGAIN</span></header>\n<section class="layout"><div class="cabinet" id="cabinet"><div class="screen-head"><span>1UP</span><span>HIGH SCORE</span><span>2UP</span></div><div class="screen"><canvas id="canvas" width="448" height="576" tabindex="0" aria-label="갤러그 게임 화면"></canvas><div id="cover"><div class="orbit">✦</div><h2>READY, FIGHTER?</h2><p>별들 사이로, 다시 출격.</p><p id="coverHint">실행엔진과 게임 데이터를 준비합니다.</p></div></div><div class="screen-foot"><span>MOVE ← →</span><span>FIRE A / SPACE</span></div></div>\n<aside><p class="step">FLIGHT CONTROL</p><h2>한 판 더,<br>그때 그 우주로.</h2><p id="status" role="status" aria-live="polite">준비 중…</p><button id="start" class="primary" disabled>게임 시작</button><div class="buttons"><button id="coin" disabled>동전 넣기 <kbd>5</kbd></button><button id="p1" disabled>1인 시작 <kbd>1</kbd></button><button id="p2" disabled>2인 시작 <kbd>2</kbd></button><button id="mute" disabled aria-pressed="false">소리 켜짐</button><button id="fullscreen">전체 화면</button></div>\n<div class="guide"><h3>출격 순서</h3><p>게임 시작 → 부팅 완료 → 동전 넣기 → 1인 시작</p><p>좌우 방향키로 이동하고 <b>A 또는 Space</b>로 발사합니다. 2인은 동전 2개를 넣고 시작하며, 같은 키로 번갈아 플레이합니다.</p><p>포획된 기체를 구출하면 듀얼 파이터로 싸울 수 있습니다.</p></div><div class="touch"><button data-scan="80" aria-label="왼쪽 이동" disabled>◀</button><button data-scan="79" aria-label="오른쪽 이동" disabled>▶</button><button data-scan="4" disabled>발사</button></div><details><summary>실행 정보 · 오류 기록</summary><label for="mode">실행 방식</label><select id="mode"><option value="js">JavaScript (기본)</option><option value="wasm">WebAssembly (성능 대안)</option></select><p>방식을 바꾸면 화면을 다시 불러옵니다.</p><pre id="log"></pre></details></aside></section>\n<footer>원작 게임: © BANDAI NAMCO Entertainment Inc. · MAME 0.239 · 사용자가 제공한 ROM으로 실행</footer></main><script src="app.js"></script></body></html>\n',encoding='utf-8')
(ROOT/'style.css').write_text(":root{color-scheme:dark;--bg:#080c18;--ink:#eef2ff;--muted:#97a4c5;--line:#27314b;--blue:#63dcff;--yellow:#ffd980}*{box-sizing:border-box}body{margin:0;background:radial-gradient(ellipse at 28% 24%,#162445 0,transparent 52%),var(--bg);color:var(--ink);font-family:Arial,'Noto Sans KR',sans-serif}main{max-width:1130px;margin:auto;padding:32px 26px}header{display:flex;justify-content:space-between;align-items:center;border-bottom:1px solid var(--line);padding-bottom:24px;margin-bottom:30px}.eyebrow,.step,.badge{font:11px monospace;letter-spacing:2px;color:var(--blue)}h1{font-size:39px;letter-spacing:4px;margin:8px 0 0}h1 span{font-size:14px;letter-spacing:0;color:var(--muted);margin-left:20px}.badge{color:var(--yellow);border:1px solid #6b5b35;padding:12px}.layout{display:grid;grid-template-columns:minmax(300px,1.2fr) 1fr;gap:60px;align-items:start}.cabinet{background:#03050a;border:1px solid #3b4b72;border-radius:20px;padding:18px 23px;box-shadow:0 20px 80px #0009,inset 0 0 0 7px #111b2c}.screen-head,.screen-foot{display:flex;justify-content:space-between;color:var(--yellow);font:12px monospace;letter-spacing:1px;padding:8px 0 16px}.screen-foot{padding:16px 0 4px;color:var(--muted);font-size:10px}.screen{position:relative;aspect-ratio:7/9;background:#000;border:1px solid #26314a;overflow:hidden}canvas{display:block;width:100%!important;height:100%!important;image-rendering:pixelated;outline:none}canvas:focus-visible{outline:2px solid var(--blue);outline-offset:-2px}#cover{position:absolute;inset:0;display:flex;flex-direction:column;justify-content:center;align-items:center;text-align:center;background:radial-gradient(ellipse,#142037,#000 75%);padding:24px}#cover[hidden]{display:none}.orbit{font-size:72px;color:var(--blue);text-shadow:0 0 40px #36c4ff}#cover h2{font:22px monospace;letter-spacing:3px}#cover p{color:var(--muted);font-size:13px;line-height:1.8}aside h2{font-size:34px;line-height:1.3;margin:16px 0 20px}#status{font-size:14px;line-height:1.7;min-height:48px;color:var(--blue)}button,select{font:inherit;border:1px solid var(--line);border-radius:9px;background:#151f34;color:var(--ink);padding:14px 16px;cursor:pointer}button:hover:enabled{border-color:var(--blue);background:#1b2e49}button:focus-visible,select:focus-visible{outline:2px solid var(--blue);outline-offset:3px}button:disabled{opacity:.4;cursor:default}.primary{width:100%;background:var(--blue);color:#071322;font-weight:bold;border:0;margin-bottom:10px}.primary:hover:enabled{background:#a4ecff;color:#071322}.buttons{display:grid;grid-template-columns:1fr 1fr;gap:9px}.buttons button{text-align:left;font-size:13px}kbd{float:right;color:var(--muted);font:12px monospace}.guide{border-top:1px solid var(--line);margin-top:25px;padding-top:10px}.guide h3{font-size:14px}.guide p{font-size:13px;line-height:1.85;color:var(--muted)}.touch{display:flex;gap:10px;margin:22px 0}.touch button{flex:1;touch-action:none;user-select:none}details{color:var(--muted);font-size:12px}summary{cursor:pointer;margin-bottom:14px}select{font-size:12px;padding:8px;margin-left:8px}pre{white-space:pre-wrap;overflow-wrap:anywhere;max-height:180px;overflow:auto;font:11px monospace;background:#0005;padding:10px}footer{color:#7e8aa8;font-size:11px;line-height:1.8;border-top:1px solid var(--line);padding-top:22px;margin-top:32px}.cabinet:fullscreen{border-radius:0;height:100vh;width:100vw;display:flex;flex-direction:column;align-items:center;padding:8px}.cabinet:fullscreen .screen{height:calc(100vh - 70px);width:auto;aspect-ratio:7/9}.cabinet:fullscreen .screen-head,.cabinet:fullscreen .screen-foot{width:min(76vh,90vw)}@media(max-width:760px){main{padding:20px 16px}.badge{display:none}header{margin-bottom:22px}.layout{grid-template-columns:1fr;gap:28px}.cabinet{max-width:470px;width:100%;margin:auto}aside h2{font-size:28px}.touch button{padding:20px}.guide{margin-top:20px}}\n",encoding='utf-8')
(ROOT/'app.js').write_text("'use strict';\nconst $=id=>document.getElementById(id), canvas=$('canvas');\nlet ready=false,started=false,failed=false,muted=false,attenuation=0,romBytes,heartbeat;\nconst held=new Set(), lines=[];\nconst mode=new URLSearchParams(location.search).get('engine')==='wasm'?'wasm':'js';\n$('mode').value=mode;\nfunction log(t){lines.push(String(t));if(lines.length>100)lines.shift();$('log').textContent=lines.join('\\n');}\nfunction status(t){$('status').textContent=t;}\nfunction fail(t){failed=true;ready=false;status('오류: '+t);log(t);$('coverHint').textContent=t;for(const id of ['start','coin','p1','p2','mute'])$(id).disabled=true;document.querySelectorAll('[data-scan]').forEach(b=>b.disabled=true);clearInterval(heartbeat);}\nfunction script(url){return new Promise((resolve,reject)=>{const e=document.createElement('script');e.src=url;e.onload=resolve;e.onerror=()=>reject(new Error('파일을 읽지 못했습니다: '+url));document.head.appendChild(e);});}\nasync function bytes(url){const r=await fetch(url,{cache:'no-store'});if(!r.ok)throw new Error(url+' 요청 실패 ('+r.status+')');return new Uint8Array(await r.arrayBuffer());}\nasync function resumeAudio(){const ctx=window.jsmame_web_audio?.get_context();if(ctx?.state==='suspended')await ctx.resume();}\nfunction key(scan,down){if(!started||failed||!window.JSMAME?.running)return;JSMAME.sdl_sendkeyboardkey(down?1:0,scan);down?held.add(scan):held.delete(scan);}\nfunction release(){for(const scan of [...held])key(scan,false);}\nfunction pulse(scan){canvas.focus();resumeAudio().catch(log);key(scan,true);setTimeout(()=>key(scan,false),120);}\n$('coin').onclick=()=>pulse(34);$('p1').onclick=()=>pulse(30);$('p2').onclick=()=>pulse(31);\n$('mute').onclick=async()=>{muted=!muted;window.jsmame_set_mastervolume(attenuation);await resumeAudio().catch(log);$('mute').textContent=muted?'소리 꺼짐':'소리 켜짐';$('mute').setAttribute('aria-pressed',String(muted));canvas.focus();};\n$('fullscreen').onclick=async()=>{try{if(document.fullscreenElement)await document.exitFullscreen();else await $('cabinet').requestFullscreen();canvas.focus();}catch(e){log('전체 화면: '+e.message);}};\n$('mode').onchange=()=>{const u=new URL(location.href);u.searchParams.set('engine',$('mode').value);location.assign(u);};\nconst scans={ArrowLeft:80,ArrowRight:79,KeyA:4,Space:44,Digit5:34,Digit1:30,Digit2:31};\nwindow.addEventListener('keydown',e=>{if(e.target.tagName==='SELECT')return;if(e.code in scans){e.preventDefault();if(started)held.add(scans[e.code]);}},true);\nwindow.addEventListener('keyup',e=>{if(e.code in scans){e.preventDefault();held.delete(scans[e.code]);}},true);\nwindow.addEventListener('blur',release);document.addEventListener('visibilitychange',()=>{if(document.hidden)release();});\nfor(const b of document.querySelectorAll('[data-scan]')){b.addEventListener('pointerdown',e=>{e.preventDefault();b.setPointerCapture(e.pointerId);resumeAudio().catch(log);canvas.focus();key(+b.dataset.scan,true);});for(const name of ['pointerup','pointercancel','lostpointercapture'])b.addEventListener(name,()=>key(+b.dataset.scan,false));}\n$('start').onclick=async()=>{if(!ready||started)return;started=true;$('start').disabled=true;$('start').textContent='실행 중';$('cover').hidden=true;status('부팅 중… 원작의 기판 검사가 끝나면 동전을 넣으세요.');canvas.focus();window.jsmame_set_mastervolume(0);await resumeAudio().catch(log);try{window.callMain(Module.arguments);}catch(e){if(e!=='unwind'&&e?.name!=='ExitStatus'){fail(e.message||String(e));return;}}let elapsed=0;heartbeat=setInterval(()=>{if(failed)return;if(window.JSMAME?.running){clearInterval(heartbeat);status('실행 중 · 타이틀 화면에서 동전을 넣고 시작하세요.');for(const id of ['coin','p1','p2','mute'])$(id).disabled=false;document.querySelectorAll('[data-scan]').forEach(b=>b.disabled=false);}else if(++elapsed>=90){clearInterval(heartbeat);fail('부팅 확인 시간이 초과되었습니다. 오류 기록을 확인하거나 WebAssembly 방식을 선택하세요.');}},1000);};\nwindow.addEventListener('error',e=>{if(e.message)fail(e.message);});\nwindow.addEventListener('unhandledrejection',e=>fail(e.reason?.message||String(e.reason)));\n(async()=>{try{\n status('준비 중 · ROM과 실행엔진을 불러옵니다.');\n romBytes=await bytes('roms/galaga.zip');\n const validation=await fetch('rom-validation.json',{cache:'no-store'});if(!validation.ok)throw new Error('ROM 검증 기록이 없습니다. Colab 1번 셀부터 실행하세요.');\n const record=await validation.json();if(!record.validated||record.driver!=='galaga')throw new Error('ROM 검증 기록이 올바르지 않습니다.');\n if(crypto.subtle){const digest=[...new Uint8Array(await crypto.subtle.digest('SHA-256',romBytes))].map(x=>x.toString(16).padStart(2,'0')).join('');if(digest!==record.sha256)throw new Error('ROM SHA-256 불일치. 1번 셀에서 다시 준비하세요.');}\n log('ROM 검증 기록 확인 · '+record.files.length+'개 구성 파일');\n const cfg=await bytes('controls.cfg');\n if(mode==='js'){status('준비 중 · JavaScript 엔진 초기화 (잠시 기다려 주세요)');await script('engine/mamegalaga.asm.js');}\n window.Module={canvas,noInitialRun:true,arguments:['galaga','-rompath','/roms','-cfg_directory','/cfg','-window','-resolution','448x576','-keepaspect','-skip_gameinfo','-samplerate','44100'],locateFile:p=>'engine/mamegalaga.wasm',print:log,printErr:t=>{log(t);if(/NOT FOUND|Required files are missing|Fatal error/.test(t))fail(t);},onAbort:t=>fail('엔진 중단: '+t),onExit:code=>{if(started&&!failed)fail('엔진이 종료되었습니다 ('+code+'). 새로고침하여 다시 시작하세요.');},preRun:[()=>{FS.mkdir('/roms');FS.mkdir('/cfg');FS.writeFile('/roms/galaga.zip',romBytes);FS.writeFile('/cfg/default.cfg',cfg);}],onRuntimeInitialized:()=>{if(failed)return;ready=true;status('준비 완료 · 게임 시작을 눌러 주세요.');$('coverHint').textContent='오른쪽의 게임 시작 버튼을 눌러 주세요.';$('start').disabled=false;log('엔진 초기화 완료 · '+mode);}};\n // Native WASM and generated JavaScript use exactly the same exported names.\n if(mode==='js')Module.instantiateWasm=(info,receive)=>{queueMicrotask(()=>{try{receive({exports:window.instantiate(info)});}catch(e){fail(e.message);}});return {};};\n await script('engine/mamegalaga.js');\n const original=window.jsmame_set_mastervolume;\n window.jsmame_set_mastervolume=db=>{attenuation=db;original(muted?-32:db);};\n }catch(e){fail(e.message+' · Colab 1→2번 셀의 완료 여부를 확인하세요.');}})();\n",encoding='utf-8')
print('HTML/CSS/JavaScript 준비 완료 — 3번 셀을 실행하세요.')


## 3. 게임 실행과 외부 접속
▶ 버튼을 누르고 **게임 열기** 버튼이 나타날 때까지 기다립니다. 로컬 웹서버 응답 → 터널 연결 로그 → 외부 주소의 실제 정상 응답을 확인한 후 버튼을 표시합니다.

같은 셀을 다시 실행하면 정상 동작 중인 서버와 터널을 재사용합니다. 다른 프로그램의 3000번 포트는 강제로 종료하지 않습니다. Cloudflare 임시 터널 연결에 실패하면 로그가 표시되며 **3번 셀만 다시 실행**하면 됩니다.

In [ ]:
# 3번 — 웹서버 실행 및 외부 접속
import importlib.util, html
from pathlib import Path
from IPython.display import display, HTML
ROOT=Path('/content/galaga_github')
if not (ROOT/'colab_support.py').is_file():raise RuntimeError('1번 셀부터 실행하세요.')
spec=importlib.util.spec_from_file_location('galaga_support',ROOT/'colab_support.py')
galaga_support=importlib.util.module_from_spec(spec);spec.loader.exec_module(galaga_support)
if '_GALAGA_STATE' not in globals():_GALAGA_STATE={}
url=galaga_support.launch(_GALAGA_STATE)
display(HTML('<a href="'+html.escape(url,quote=True)+'" target="_blank" rel="noopener" style="display:inline-block;padding:15px 28px;border-radius:10px;background:#1663e8;color:white;text-decoration:none;font-weight:bold">게임 열기 ↗</a>'))
print('외부 접속 주소:',url)
print('게임 시작 → 기판 검사 대기 → 동전 넣기 → 1인 시작')
print('외부 접속을 유지하려면 Colab 런타임 연결을 유지하세요.')


## 조작과 게임 방식

| 동작 | 키 / 버튼 |
|---|---|
| 이동 | ← / → |
| 발사 | A / Space |
| 동전 | 5 / 동전 넣기 |
| 1인 시작 | 1 / 1인 시작 |
| 2인 시작 | 2 / 2인 시작 — 동전 2개, 같은 키로 교대 |
| 터치 | 화면의 ◀ / ▶ / 발사 버튼 |
| 음소거·복원 | 소리 켜짐 / 소리 꺼짐 |
| 전체 화면 | 전체 화면 버튼 |

적 편대를 처치하고 급강하 공격을 피합니다. 보스의 빔에 포획된 기체를 구출하면 듀얼 파이터가 됩니다. 챌린징 스테이지, 점수, 목숨, 게임오버는 원작 ROM이 처리합니다. 사운드는 MAME에서 생성하며 별도 음악을 겹쳐 재생하지 않습니다.

## 확인 범위와 제한

직접 확인: JavaScript 엔진 변환과 브라우저 초기화, 시작 버튼에서 ROM 검사 경로 진입, 필수 ROM 16개 요구 목록, 파일 해시 검사 및 손상 데이터 거부, HTTP 압축 전송과 파일 공개 범위, 노트북/코드 문법.

**ROM이 제공되지 않아 실제 플레이(동전·이동·발사·점수·스테이지), 소리 청취·음소거 복원은 검증하지 못했습니다. 사용자의 Colab 환경에서 전체 실행도 미검증입니다.** 제작 환경의 네트워크 제한으로 Cloudflare 외부 터널 연결도 확인하지 못했습니다. 로컬 서버와 재실행 시 서버 재사용은 확인했습니다. 세부 결과는 ZIP의 `VERIFICATION.txt`를 확인하세요.

출처: [Bandai Namco 공식 설명](https://galaga.com/en/history/galaga.php), [MAME 0.239](https://github.com/mamedev/mame/tree/mame0239), [Emularity 엔진](https://archive.org/details/emularity_engine_v1), [Binaryen](https://github.com/WebAssembly/binaryen/releases/tag/version_133), [Cloudflared](https://github.com/cloudflare/cloudflared/releases/tag/2026.10.2). 상세 출처·라이선스는 ZIP에 포함했습니다.
